# NB-16 — ε₅₀ on the replicate checkpoints

**NB-12's queue runner, pointed at the NB-14 replicates.** The attack machinery, the front end,
the PGD loop, the bisection and the per-batch resume are byte-identical. Only `QUEUE` changed.

## What this buys

The paper reports one between-run difference: 75.0% against 41.0% at the 1-hour rung, 34 pp,
from two runs launched at the same seed. One pair gives a difference, not a spread, and the
obvious reviewer question — *is 34 pp typical or did you get unlucky?* — cannot be answered
from it.

This run attacks the deliberate replicates, so the between-run standard deviation can be
estimated at three rungs instead of inferred from one pair. That turns the paper's power
analysis from an extrapolation into a measurement.

## Stage 1 — the queue (≈4.6 GPU-h per condition)

| # | condition | what it buys |
|---|---|---|
| 1 | `ckpt-xlsr-1h-s1338/final` | 1 h gets a third run -> σ with 2 df at one rung |
| 2 | `ckpt-xlsr-1h-s1339/final` | 1 h gets a fourth -> σ with 3 df |
| 3 | `ckpt-xlsr-5h-s1338/final`  | spread at a second rung |
| 4 | `ckpt-xlsr-25h-s1338/final` | spread at a third rung |

Ordered by value. At 4.6 GPU-h each and `WALL_HOURS = 11`, **two conditions per session** is
what fits. Run it twice, or run two copies concurrently with two conditions each.

## Stage 2 — analysis in the same session

* ASR@2×10⁻³, censoring and ceiling-imputed median for each new condition
* **every pairwise exact McNemar within each rung**, on the shared utterances
* **pooled between-run σ with its confidence interval**, and the required runs per condition
  that follows from it — reported as an interval, because σ from a handful of runs is imprecise
  and a point estimate here would be false precision
* refreshed ladder and pair tables, if the Task B CSVs are attached
* a paste-ready RUNLOG block

## Attach

**Required:** the replicate checkpoints in this session's `QUEUE`, `bn-asr-splits`,
`bengaliai-speech` (Competitions → "bengaliai speech"), and any one `eps50-*-n100` dataset
for the frozen targets.

**Attach all six Task B `eps50-*-n100` datasets plus `eps50-rung1-trajfinal-t5-n100`.**
Without them Stage 2 can only see this session's own conditions, and the whole point is to
compare the new runs against the published ones at the same rung.

On a re-commit, also attach the previous session's output: completed utterances are skipped,
not redone.

GPU T4 x2. Internet off is fine.

## The honest warning

4.6 GPU-h per condition against Kaggle's 12-hour kill means **a two-condition session is
expected to finish and a four-condition session is not.** `WALL_HOURS = 11` stops cleanly,
results are written after every batch, and Stage 2 runs on whatever exists. A summary reading
`partial` or `not-started` is a normal outcome, not a failure.


## 1 · Queue and locked attack configuration

In [ ]:
NOTEBOOK = "NB-16-eps50-replicates"

# ---------------------------------------------------------------------------
# THE QUEUE. `tag` lands in the output filename as eps50_{tag}_T5_n100.csv and
# is what Stage 2 groups by, so keep the rung{N}-s{SEED} shape -- Stage 2 parses
# it. Comment out what this session should not attempt: at 4.6 GPU-h each and
# WALL_HOURS=11, two conditions fit and four do not.
#
# Three sessions of two, one rung each, so a session that dies loses one rung's
# pair rather than half of every rung:
#   Session A: conditions 1-2 (the 1 h pair)   <- as shipped
#   Session B: conditions 3-4 (the 5 h pair)
#   Session C: conditions 5-6 (the 25 h pair)
# Or run two copies concurrently with one session's pair each.
#
# Keep each RUNG's two runs in the SAME session. They are the clean
# within-environment pair -- both trained on the current transformers, where the
# published runs used 4.44.2 -- so they are the comparison sigma should rest on,
# and measuring them under one attack session keeps that true on the attack side
# too.
# ---------------------------------------------------------------------------
QUEUE = [
    dict(tag="rung1-s1338",  ckpt_dataset="ckpt-xlsr-1h-s1338",  rung=1,  checkpoint="final"),
    dict(tag="rung1-s1339",  ckpt_dataset="ckpt-xlsr-1h-s1339",  rung=1,  checkpoint="final"),
#   dict(tag="rung5-s1338",  ckpt_dataset="ckpt-xlsr-5h-s1338",  rung=5,  checkpoint="final"),
#   dict(tag="rung5-s1339",  ckpt_dataset="ckpt-xlsr-5h-s1339",  rung=5,  checkpoint="final"),
#   dict(tag="rung25-s1338", ckpt_dataset="ckpt-xlsr-25h-s1338", rung=25, checkpoint="final"),
#   dict(tag="rung25-s1339", ckpt_dataset="ckpt-xlsr-25h-s1339", rung=25, checkpoint="final"),
]

WALL_HOURS = 11.0        # hard stop for the WHOLE session. Kaggle kills at 12.
MIN_HOURS_TO_START = 1.0 # do not begin a condition with less than this left; a
                         # condition abandoned at 3% wastes the load and the setup

# --- Task B's configuration, reproduced exactly. Do not tune. ----------------
EDIT_TYPE      = "T5"
N_EVAL         = 100
BATCH_SIZE     = 8
USE_AMP        = True
FIXED_EPS      = None
EPS_LO, EPS_HI = 1e-4, 2e-3
BISECT_ROUNDS  = 5
PROBE_STEPS    = 400
CONFIRM_STEPS  = 800
STALL_PATIENCE = 100
CHECK_EVERY    = 50
LR             = 1e-4
SEED           = 1337
VALIDATE_BATCHING = False

_LOCKED = dict(edit=EDIT_TYPE, n=N_EVAL, bs=BATCH_SIZE, amp=USE_AMP, fx=FIXED_EPS,
               lo=EPS_LO, hi=EPS_HI, br=BISECT_ROUNDS, ps=PROBE_STEPS,
               cs=CONFIRM_STEPS, sp=STALL_PATIENCE, ce=CHECK_EVERY, lr=LR, seed=SEED)
assert _LOCKED == dict(edit="T5", n=100, bs=8, amp=True, fx=None, lo=1e-4, hi=2e-3,
                       br=5, ps=400, cs=800, sp=100, ce=50, lr=1e-4, seed=1337), (
    "Attack settings were edited. Every number here is only interpretable against "
    "Task B's six runs if the configuration matches them exactly.")
assert EDIT_TYPE == "T5", (
    "T1 cannot run on this eval set. Only 3 of these 100 utterances carry a "
    "standalone negation particle, and none of the original 25 do, so a paired "
    "T1-vs-T5 comparison is impossible here. Worse, task_b_targets_n100.csv has no "
    "T1 target column at all -- removing this assertion would attack the T5 targets "
    "while labelling the output T1. T1 lives on NB-03's 38-utterance spine; see NB-13.")
assert len({c["tag"] for c in QUEUE}) == len(QUEUE), "duplicate tag in QUEUE"

# Every tag must be parseable by Stage 2, or the condition is measured and then
# silently dropped from the variance table -- 4.6 GPU-h for nothing.
import re as _re
for _c in QUEUE:
    _m = _re.fullmatch(r"rung(\d+)-s(\d+)", _c["tag"])
    assert _m, (f"tag {_c['tag']!r} does not match rung{{N}}-s{{SEED}}. Stage 2 parses "
                "the tag to group runs by rung; an unparseable tag is a condition that "
                "costs 4.6 GPU-h and never reaches the variance table.")
    assert int(_m.group(1)) == _c["rung"], (
        f"tag {_c['tag']!r} says rung {_m.group(1)} but rung={_c['rung']}")
    assert _c["ckpt_dataset"] == f"ckpt-xlsr-{_c['rung']}h-s{_m.group(2)}", (
        f"tag {_c['tag']!r} and ckpt_dataset {_c['ckpt_dataset']!r} disagree. This is the "
        "mislabelling that attacks one model and names the output after another.")

print(f"NB-16 replicate queue | {len(QUEUE)} conditions | wall clock {WALL_HOURS} h")
for i, c in enumerate(QUEUE, 1):
    print(f"  {i}. {c['tag']:20s} {c['ckpt_dataset']}/{c['checkpoint']}")
print("\nPublished runs, same targets, same 100 utterances, ASR@2e-3:")
for k, v in [("1h  s1337 final",75),("1h  s1337 run2",41),("5h  s1337 final",79),
             ("25h s1337 final",66),("100h ms-1000",37),("100h ms-2000",46),
             ("100h ms-3500",53)]:
    print(f"    {k:16s} {v}%")
print("  The 1 h rung already has two runs 34 pp apart at the SAME seed. The")
print("  replicates in this queue turn that pair into a distribution.")

## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata, gc
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
# Project-wide Bengali normaliser. Defined HERE, in boilerplate, because the
# input-resolution cell asserts with it and in NB-05 it lived in the (later)
# target-construction cell -- which a queue runner does not have.
ZW = dict.fromkeys(map(ord, "\u200c\u200d"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# `jiwer` is deliberately NOT imported: it is not preinstalled on Kaggle, and
# the only place this notebook would use it is a FIXED_EPS clean-WER branch
# that cannot execute (FIXED_EPS is None here) and already guards itself.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")
PROV["transformers"] = transformers.__version__
PROV["numpy"] = numpy.__version__


## 3 · Inputs, eval set and frozen targets

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, ("Attach bn-asr-splits -- test.csv is needed to resolve audio paths "
                "for the 100 target utterances.\n"
                f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
print("splits:", SPLITS)

# ---------------------------------------------------------------------------
# EVAL SET AND TARGETS: read, never regenerated.
#
# Task B's targets are published inside every eps50-*-n100 dataset as
# task_b_targets_n100.csv. Reading them makes this run byte-comparable by
# construction rather than by reproducing a generator. The hash is checked.
# ---------------------------------------------------------------------------
TGT_CSV = next((f"{p}/task_b_targets_n100.csv" for p in inputs
                if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT_CSV, ("task_b_targets_n100.csv not found. Attach ONE Task B result "
                 "dataset, e.g. zoayriaabedin/eps50-rung1-final-t5-n100.\n"
                 f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
TGT_SHA = hashlib.sha256(open(TGT_CSV, "rb").read()).hexdigest()
print("targets:", TGT_CSV)
print("targets sha256:", TGT_SHA)
assert TGT_SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "These are NOT the targets the six Task B runs used. The comparison against "
    "run 1's 75% would be meaningless.")
tg = pd.read_csv(TGT_CSV)
assert len(tg) == 100 and tg.utt_id.nunique() == 100

_full = pd.read_csv(f"{SPLITS}/test.csv")
# source_tokens / target_tokens are carried through: the results writer uses
# both, and eps50_per_token divides by target_tokens. Dropping them here would
# KeyError at save time, after the whole sweep had run.
ev = tg[["utt_id", "target_text", "source_tokens", "target_tokens"]].merge(
        _full, on="utt_id", how="left")
assert ev["path"].notna().all(), "some target utterances are absent from test.csv"
assert (ev.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
EV_CSV = TGT_CSV
PROV["attack_eval_csv"] = os.path.basename(TGT_CSV)
PROV["targets_sha256"] = TGT_SHA
PROV["fixed_eps"] = FIXED_EPS
PROV["bisect_rounds"] = BISECT_ROUNDS
# The checkpoint dataset and model_dir are PER CONDITION here, so they are not
# recorded once at the top: every result row carries its own `model_dir`, and the
# queue summary records the tags.
for _c in ("path","text","target_text","source_tokens","target_tokens","duration"):
    assert _c in ev.columns, f"ev is missing {_c}, which a later cell needs"
print(f"eval set: {len(ev)} utterances, manifest order (original 25 first)")

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

ev, AUDIO_ROOT = resolve_audio(ev)
absent = [p for p in ev["path"] if not os.path.exists(p)]
assert not absent, f"{len(absent)}/{len(ev)} attack audio files missing, e.g.\n  {absent[0]}"
print(f"audio root: {AUDIO_ROOT}")
print(f"attack set: {len(ev)} utterances, all present")

## 4 · Front end, loss, PGD, bisection

Byte-identical to NB-05. These read `model`, `processor`, `BLANK_ID` and `DO_NORMALIZE` as globals at call time, which is what lets Stage 1 swap checkpoints without redefining anything.

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Per-condition loader

In [ ]:
def load_condition(cfg):
    """Resolve and load one checkpoint, rebinding the globals the attack closures read.

    The closures in the cells above look up `model`, `processor`, `BLANK_ID` and
    `DO_NORMALIZE` at CALL time, not definition time, so rebinding them here is
    all that is needed to point the whole machine at a different checkpoint.
    Every one of the four must be rebound: leaving a stale BLANK_ID or
    DO_NORMALIZE behind would silently attack the new model through the old
    model's decoding assumptions.
    """
    global model, processor, BLANK_ID, DO_NORMALIZE, MODEL_DIR, RUNG, CHECKPOINT
    ds = cfg["ckpt_dataset"]
    cands = [p for p in inputs if re.search(rf"{re.escape(ds)}$", os.path.basename(p))]
    assert len(cands) == 1, (
        f"Need exactly one input matching '{ds}', found {len(cands)}: "
        f"{[os.path.basename(c) for c in cands]}")
    ck = cands[0]
    if cfg["checkpoint"] == "final":
        MODEL_DIR = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    else:
        MODEL_DIR = f"{ck}/milestones/step-{cfg['checkpoint'].split('-')[1]}"
    assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"

    processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.config.apply_spec_augment = False
    DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
    BLANK_ID = model.config.pad_token_id
    RUNG, CHECKPOINT = cfg["rung"], cfg["checkpoint"]
    print(f"  loaded {MODEL_DIR}")
    print(f"  do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")
    return MODEL_DIR


def unload():
    global model, processor
    try:
        del model, processor
    except NameError:
        pass
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6 · Per-condition sweep

In [ ]:
def run_condition(cfg, deadline):
    """Run one condition to completion or to the deadline.

    Returns (status, rows). status is "complete", "partial" or "skipped".
    Results are written after EVERY batch, so a kill at any point loses at most
    one batch.
    """
    tag = cfg["tag"]
    OUT = f"{WORK}/eps50_{tag}_{EDIT_TYPE}_n{N_EVAL}.csv"

    # ---------------------------------------------------------------------
    # Resume: this session's own output first, then ANY attached source
    # carrying the same filename, searched at EVERY mount depth.
    #
    # The original globbed only `/kaggle/input/*/F` and `/kaggle/input/*/*/F`.
    # On Kaggle's current layout a dataset file lives at
    #     /kaggle/input/datasets/<owner>/<name>/F
    # which is FOUR components deep, so neither pattern could ever match it --
    # the resume path was dead code, and a re-commit would silently re-measure
    # all 100 utterances instead of the 20 outstanding. Stage 2's own discovery
    # already searched to that depth; the resume did not, and no run had
    # exercised it, so nothing caught the mismatch.
    #
    # A notebook-output attachment may mount at a different depth again, so this
    # sweeps 1 to 4 and PRINTS every source it used. A silent zero is the failure
    # mode that matters here, so the no-source case says so loudly.
    # ---------------------------------------------------------------------
    results = []
    _base = os.path.basename(OUT)
    srcs = [OUT] if os.path.exists(OUT) else []
    for _d in range(1, 5):
        srcs += sorted(glob.glob("/kaggle/input/" + "*/" * _d + _base))
    srcs = list(dict.fromkeys(srcs))          # keep order, drop repeats
    seen = set()
    for p in srcs:
        _n = 0
        for r in pd.read_csv(p).to_dict("records"):
            if r["utt_id"] not in seen:
                seen.add(r["utt_id"]); results.append(r); _n += 1
        print(f"  resume source: +{_n:3d} new rows  {p}")
    if results:
        print(f"  resuming: {len(results)} utterances already measured")
    else:
        print(f"  NO RESUME SOURCE for {_base} -- starting from scratch.")
        print("    If you attached a previous partial run and expected to continue,")
        print("    stop and check it is visible: the previous session's Stage 2")
        print("    listing prints the exact filename this looks for.")
    if len(results) >= len(ev):
        print(f"  ALREADY COMPLETE ({len(results)}/{len(ev)}) -- skipping")
        pd.DataFrame(results).to_csv(OUT, index=False)
        return "skipped", results

    todo = ev[~ev.utt_id.isin(seen)].reset_index(drop=True)
    batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    print(f"  {len(todo)} remaining -> {len(batches)} batches of up to {BATCH_SIZE}")

    for bi, sub in enumerate(batches):
        if time.time() > deadline:
            print(f"  *** wall clock reached, {sum(len(b) for b in batches[bi:])} "
                  f"utterances unmeasured in this condition ***")
            return "partial", results
        xb, mb = load_batch(sub)
        tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
              for t in sub.target_text]
        tl = torch.tensor([len(t) for t in tp], device=DEVICE)
        tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
        t0 = time.time()
        eps, width, conf, best = eps50_batch(xb, mb, tpad, tl, list(sub.target_text),
                                             verbose=False)
        dt = time.time() - t0
        x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
        snrs = snr_db_b(xb, x_adv, mb)
        hyp = decode_b(x_adv, mb)
        for i, (_, r) in enumerate(sub.iterrows()):
            ok = not np.isnan(eps[i])
            results.append(dict(
                utt_id=r["utt_id"], tag=tag, rung=RUNG, edit_type=EDIT_TYPE,
                checkpoint=CHECKPOINT, duration=r["duration"], source_text=r["text"],
                target_text=r["target_text"], source_tokens=int(r["source_tokens"]),
                target_tokens=int(r["target_tokens"]),
                eps50=float(eps[i]) if ok else None, censored=not ok,
                eps50_per_token=(float(eps[i]) / r["target_tokens"]) if ok else None,
                bracket_width=float(width[i]), snr_db=float(snrs[i]),
                decoded=(hyp[i] if ok else None),
                exact_match=(bool(hyp[i] == r["target_text"]) if ok else False),
                use_amp=USE_AMP, model_dir=MODEL_DIR))
        pd.DataFrame(results).to_csv(OUT, index=False)
        rem = len(batches) - (bi + 1)
        eta = time.time() + rem * dt
        print(f"    batch {bi+1}/{len(batches)}  {dt/60:4.1f} min  "
              f"{int((~np.isnan(eps)).sum())}/{len(sub)} uncensored  "
              f"{len(results)}/{len(ev)} rows"
              + (f"  | {rem} left, ~{rem*dt/3600:.1f} h"
                 f" ({'fits' if eta < deadline else 'WILL NOT FIT'})" if rem else ""))
    return ("complete" if len(results) >= len(ev) else "partial"), results

## 7 · STAGE 1 — run the queue

In [ ]:
SESSION_START = time.time()
DEADLINE = SESSION_START + WALL_HOURS * 3600
summary = []

for qi, cfg in enumerate(QUEUE, 1):
    left = (DEADLINE - time.time()) / 3600
    print("\n" + "=" * 74)
    print(f"[{qi}/{len(QUEUE)}] {cfg['tag']}   ({left:.1f} h of wall clock left)")
    print("=" * 74)
    if left < MIN_HOURS_TO_START:
        print(f"  only {left:.1f} h left, below MIN_HOURS_TO_START={MIN_HOURS_TO_START}.")
        print("  NOT STARTING. Publish /kaggle/working, attach it back, re-commit.")
        summary.append(dict(tag=cfg["tag"], status="not-started", n=0, asr=None))
        continue
    try:
        load_condition(cfg)
        status, rows = run_condition(cfg, DEADLINE)
    except Exception as e:
        print(f"  !! {type(e).__name__}: {e}")
        summary.append(dict(tag=cfg["tag"], status=f"error: {type(e).__name__}",
                            n=0, asr=None))
        unload(); continue
    df = pd.DataFrame(rows)
    asr = float((~df.censored.astype(bool)).mean() * 100) if len(df) else None
    summary.append(dict(tag=cfg["tag"], status=status, n=len(df), asr=asr))
    print(f"  {status.upper()}  {len(df)}/{len(ev)} measured"
          + (f"  ASR@2e-3 = {asr:.1f}%" if asr is not None else ""))
    unload()

print("\n" + "=" * 74)
print(f"SESSION DONE in {(time.time()-SESSION_START)/3600:.2f} h")
print("=" * 74)

## 8 · Queue summary and provenance

In [ ]:
sm = pd.DataFrame(summary)
print(sm.to_string(index=False))

# Published ASR per rung at seed 1337, for an immediate read before Stage 2.
PUB = {1: [("s1337 final", 75.0), ("s1337 run2", 41.0)],
       5: [("s1337 final", 79.0)], 25: [("s1337 final", 66.0)]}
print("\nAgainst the published runs, same targets, same 100 utterances, ASR@2e-3:")
for r in sm.itertuples():
    if r.asr is None: continue
    rung = int(re.fullmatch(r"rung(\d+)-s\d+", r.tag).group(1))
    print(f"  rung {rung} h:")
    for lbl, v in PUB.get(rung, []):
        print(f"    {lbl:12s} {v:5.1f}%")
    print(f"    {r.tag:12s} {r.asr:5.1f}%   <- NEW")
    spread = [v for _, v in PUB.get(rung, [])] + [r.asr]
    if len(spread) > 1:
        print(f"    range across {len(spread)} runs of this rung: "
              f"{min(spread):.1f}-{max(spread):.1f}%  ({max(spread)-min(spread):.1f} pp)")
print("\nRead these against the pair effects they are meant to explain: 38 pp (40x),")
print("33 pp (16x), 13 pp (4x). Stage 2 does the pairing and the sigma properly.")

PROV.update(notebook="NB-16-eps50-replicates", queue=[c["tag"] for c in QUEUE],
            summary=summary, wall_hours=WALL_HOURS,
            session_hours=round((time.time()-SESSION_START)/3600, 2),
            n_eval=int(len(ev)), edit_type=EDIT_TYPE, seed=SEED,
            bisect_rounds=BISECT_ROUNDS, probe_steps=PROBE_STEPS,
            confirm_steps=CONFIRM_STEPS, batch_size=BATCH_SIZE, lr=LR,
            eps_lo=EPS_LO, eps_hi=EPS_HI, use_amp=USE_AMP)
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False, indent=2,
          default=str)
json.dump({"title": "eps50-replicates-n100", "id": "YOURUSERNAME/eps50-replicates-n100",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print(f"\nwrote {len([f for f in os.listdir(WORK) if f.endswith('.csv')])} CSVs to {WORK}")
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nCHECK THIS FILE LIST before publishing: publishing replaces the dataset and")
print("deletes anything a partial run did not regenerate.")
if any(s["status"] in ("partial", "not-started") for s in summary):
    print("\n*** NOT EVERYTHING FINISHED. Publish, attach the output back as an input,")
    print("    and re-commit: completed utterances are skipped, not redone. ***")

## 9 · STAGE 2 — statistics

Runs on whatever Stage 1 produced, plus any Task B CSVs that happen to be attached. Exact tests
are implemented inline rather than imported, so nothing here depends on a package version.

In [ ]:
from math import comb
CEIL = float(EPS_HI)

def binom_two_sided(k, n):
    """Exact two-sided binomial test at p=0.5, method of small p-values."""
    if n == 0:
        return 1.0
    pmf = [comb(n, i) * 0.5 ** n for i in range(n + 1)]
    tol = pmf[k] * (1 + 1e-9)
    return min(1.0, sum(v for v in pmf if v <= tol))

def mcnemar(a, b):
    """a, b: dicts utt_id -> success bool, over the SAME utterances.
    Returns (p, n10, n01, n_common). Paired, exact."""
    ids = sorted(set(a) & set(b))
    n10 = sum(1 for u in ids if a[u] and not b[u])
    n01 = sum(1 for u in ids if b[u] and not a[u])
    return binom_two_sided(min(n10, n01), n10 + n01), n10, n01, len(ids)

def med(v):
    v = sorted(v); n = len(v)
    return None if not n else (v[n // 2] if n % 2 else 0.5 * (v[n//2 - 1] + v[n//2]))

def load_eps(path):
    """-> (utt_id -> success bool, utt_id -> imputed eps50)"""
    d = pd.read_csv(path)
    cens = d.censored.astype(str).str.lower().isin(["true", "1", "yes"])
    succ = {r.utt_id: (not c) for r, c in zip(d.itertuples(), cens)}
    eps = {r.utt_id: (CEIL if c else float(r.eps50))
           for r, c in zip(d.itertuples(), cens)}
    return succ, eps

# ---- gather every eps50 CSV we can see: this session's, and attached ---------
#
# Keyed on basename, and two attached datasets CAN carry the same basename.
# Task B named each run's output from the rung and the checkpoint, so run 1
# (ckpt-xlsr-1h/final, ASR 75%) and run 7 (ckpt-xlsr-1h-traj/final, the SAME
# traj model, ASR 61%) both produced `eps50_rung1_final_T5_n100.csv`. A
# setdefault would take whichever mounted first and nothing downstream would
# know which model the "1h final" row describes.
#
# This session's own output wins deliberately -- it is first in the pattern list
# and is the one measurement whose provenance is certain. Any OTHER duplicate is
# printed with its ASR, so a shadowed file is visible rather than silent.
_paths = {}
for pat in (f"{WORK}/eps50_*.csv",
            "/kaggle/input/*/eps50_*.csv", "/kaggle/input/*/*/eps50_*.csv",
            "/kaggle/input/*/*/*/eps50_*.csv"):
    for p in glob.glob(pat):
        b = os.path.basename(p)
        _paths.setdefault(b, [])
        if p not in _paths[b]:
            _paths[b].append(p)

FOUND = {k: v[0] for k, v in _paths.items()}

_dupes = {k: v for k, v in _paths.items() if len(v) > 1}
if _dupes:
    print("*** SAME FILENAME IN MORE THAN ONE LOCATION -- check these ***")
    for k, v in _dupes.items():
        print(f"  {k}")
        for q in v:
            try:
                _d = pd.read_csv(q, usecols=["censored"])
                _c = _d.censored.astype(str).str.lower().isin(("true", "1", "yes"))
                print(f"     ASR {100.0*(~_c).mean():5.1f}%  n={len(_d)}  "
                      f"{'USED' if q == FOUND[k] else 'ignored'}   {q}")
            except Exception as e:
                print(f"     unreadable ({type(e).__name__})  {q}")
    print("  If two of these are DIFFERENT measurements, detach the one you do not")
    print("  want. Stage 2 below uses only the one marked USED.\n")

print(f"{len(FOUND)} eps50 CSVs visible:")
for k in sorted(FOUND):
    print("   ", k)

In [ ]:
# ---- per-condition summary of what Stage 1 produced ------------------------
print("\n" + "=" * 74)
print("STAGE 2a  --  the new conditions")
print("=" * 74)
new_rows = []
for cfg in QUEUE:
    f = f"eps50_{cfg['tag']}_{EDIT_TYPE}_n{N_EVAL}.csv"
    if f not in FOUND:
        print(f"  {cfg['tag']:20s} no output -- did not run")
        continue
    succ, eps = load_eps(FOUND[f])
    n = len(succ); k = sum(succ.values())
    ok = [eps[u] for u in succ if succ[u]]
    new_rows.append(dict(tag=cfg["tag"], n=n, asr=100.0 * k / n if n else None,
                         censored=n - k, eps50_ok=med(ok), eps50_imp=med(list(eps.values()))))
    print(f"  {cfg['tag']:20s} n={n:3d}  ASR@{CEIL:.0e} = {100*k/n:5.1f}%  "
          f"censored {n-k:3d}  eps50(ok) {med(ok) if ok else float('nan'):.4e}  "
          f"imputed {med(list(eps.values())):.4e}")
if new_rows:
    pd.DataFrame(new_rows).to_csv(f"{WORK}/stage2_new_conditions.csv", index=False)

In [ ]:
# ---- STAGE 2b --- BETWEEN-RUN VARIANCE, the question this run exists to answer
print("\n" + "=" * 74)
print("STAGE 2b  --  BETWEEN-RUN VARIANCE")
print("=" * 74)

# Every run of a rung's FINAL checkpoint that we might be able to see, named
# explicitly. The published Task B filenames do not share a shape with the
# replicate tags (`rung1_final` vs `rung1-s1338`), and a pattern that silently
# failed to match one of them would drop a run out of sigma -- which would make
# the paper MORE confident on LESS evidence. So they are listed, not globbed.
RUN_INDEX = [
    (1,  "s1337 final", "eps50_rung1_final_T5_n100.csv"),
    (1,  "s1337 run2",  "eps50_rung1-trajfinal_T5_n100.csv"),
    (5,  "s1337 final", "eps50_rung5_final_T5_n100.csv"),
    (25, "s1337 final", "eps50_rung25_final_T5_n100.csv"),
]
for _r in (1, 5, 25):
    for _s in (1338, 1339, 1340, 1341):
        RUN_INDEX.append((_r, f"s{_s}", f"eps50_rung{_r}-s{_s}_{EDIT_TYPE}_n{N_EVAL}.csv"))

RUNS = {}
for rung, label, fn in RUN_INDEX:
    if fn not in FOUND:
        continue
    succ, eps = load_eps(FOUND[fn])
    if not succ:
        continue
    RUNS.setdefault(rung, []).append(dict(
        label=label, file=fn, succ=succ, eps=eps, n=len(succ),
        asr=100.0 * sum(succ.values()) / len(succ)))

if not RUNS:
    print("  no rung-final eps50 CSVs visible at all. Nothing to do here.")
else:
    for rung in sorted(RUNS):
        print(f"\n  rung {rung} h  --  {len(RUNS[rung])} run(s) visible")
        for r in RUNS[rung]:
            print(f"    {r['label']:12s} n={r['n']:3d}  ASR@{CEIL:.0e} = {r['asr']:5.1f}%  "
                  f"censored {r['n'] - sum(r['succ'].values()):3d}")

# ---- pairwise exact McNemar within each rung --------------------------------
print("\n" + "-" * 74)
print("PAIRWISE McNEMAR WITHIN EACH RUNG  --  paired on the shared utterances")
print("-" * 74)
print("  Two runs of the SAME condition. Under the null that retraining changes")
print("  nothing, every one of these should be non-significant.")
pair_rows = []
for rung in sorted(RUNS):
    rs = RUNS[rung]
    if len(rs) < 2:
        print(f"\n  rung {rung} h: only one run -- no within-rung pair yet")
        continue
    print(f"\n  rung {rung} h:")
    for i in range(len(rs)):
        for j in range(i + 1, len(rs)):
            a, b = rs[i], rs[j]
            pv, n10, n01, nc = mcnemar(a["succ"], b["succ"])
            d = b["asr"] - a["asr"]
            pair_rows.append(dict(rung=rung, run_a=a["label"], run_b=b["label"],
                                  asr_a=round(a["asr"], 1), asr_b=round(b["asr"], 1),
                                  delta_pp=round(d, 1), n10=n10, n01=n01, n=nc,
                                  mcnemar_p=pv))
            print(f"    {a['label']:12s} vs {b['label']:12s} "
                  f"{a['asr']:5.1f}% vs {b['asr']:5.1f}%  "
                  f"delta {d:+6.1f} pp  discordant {n10}/{n01}  "
                  f"p = {pv:.3e}{'  SIGNIFICANT' if pv < 0.05 else ''}")
if pair_rows:
    pd.DataFrame(pair_rows).to_csv(f"{WORK}/stage2_within_rung_pairs.csv", index=False)

# ---- pooled between-run sigma, with an interval -----------------------------
print("\n" + "-" * 74)
print("POOLED BETWEEN-RUN SIGMA IN ATTACK SUCCESS")
print("-" * 74)

# Two-sided chi-square quantiles, embedded so Stage 2 keeps its promise of
# depending on no package version. df > 12 falls back to scipy if present.
CHI2 = {1: (0.000982, 5.0239), 2: (0.05064, 7.3778), 3: (0.21580, 9.3484),
        4: (0.48442, 11.1433), 5: (0.83121, 12.8325), 6: (1.23734, 14.4494),
        7: (1.68987, 16.0128), 8: (2.17973, 17.5345), 9: (2.70039, 19.0228),
        10: (3.24697, 20.4832), 11: (3.81575, 21.9200), 12: (4.40379, 23.3367)}

def chi2_bounds(df):
    if df in CHI2:
        return CHI2[df]
    try:
        from scipy.stats import chi2
        return float(chi2.ppf(0.025, df)), float(chi2.ppf(0.975, df))
    except Exception:
        return None, None

num = den = 0.0
contrib = []
for rung in sorted(RUNS):
    v = [r["asr"] for r in RUNS[rung]]
    if len(v) < 2:
        continue
    s2 = float(np.var(v, ddof=1))
    num += (len(v) - 1) * s2
    den += (len(v) - 1)
    contrib.append((rung, len(v), float(np.std(v, ddof=1)), min(v), max(v)))
    print(f"  rung {rung:3d} h: {len(v)} runs  ASR {min(v):5.1f}-{max(v):5.1f}%  "
          f"sd {np.std(v, ddof=1):5.1f} pp")

PAIR_EFFECTS = {1: ("40x", 38.0), 5: ("16x", 33.0), 25: ("4x", 13.0)}
sigma = sigma_raw = req_lo = req_hi = req = None
if den <= 0:
    print("\n  No rung has two runs yet, so there is no sigma to pool.")
    print("  This is the entire purpose of the queue -- finish it before writing.")
else:
    sigma_raw = (num / den) ** 0.5
    df = int(den)

    # BIAS CORRECTION. E[s] = c4(df) * sigma, and c4(1) = 0.7979, so the sample
    # sd understates sigma badly at tiny df. The paper's sec:power quotes ~30 pp
    # from the two 1 h runs by inverting the expected range for n=2
    # (E[range] = 1.128 sigma); 34/1.128 = 30.1, and (34/sqrt2)/0.7979 = 30.1 too.
    # The corrected value is what the paper reports, so it is what this notebook
    # must report -- otherwise the manuscript and its own notebook disagree.
    # The correction is <5% by df 5. NB-17 does this identically.
    def c4(d):
        from math import lgamma, exp, log
        return exp(0.5 * log(2.0 / d) + lgamma((d + 1) / 2.0) - lgamma(d / 2.0))

    _c4 = c4(df)
    sigma = sigma_raw / _c4
    lo_q, hi_q = chi2_bounds(df)
    print(f"\n  pooled sd (raw)      = {sigma_raw:.2f} pp   (df = {df}, "
          f"from {len(contrib)} rung(s) with >=2 runs)")
    print(f"  bias-corrected sigma = {sigma:.2f} pp   (/ c4({df}) = {_c4:.4f})  "
          "<- REPORT THIS ONE")
    if lo_q:
        # The chi-square interval is for sigma itself and takes the raw sd.
        s_lo = sigma_raw * (df / hi_q) ** 0.5
        s_hi = sigma_raw * (df / lo_q) ** 0.5
        print(f"  95% CI for sigma     = [{s_lo:.2f}, {s_hi:.2f}] pp")
        print("\n  That interval is wide, and reporting the point estimate alone would be")
        print("  false precision: sigma from a handful of runs is a very noisy statistic.")
    else:
        s_lo = s_hi = None

    print("\n  Against the effects it has to explain:")
    for rung, (ex, eff) in PAIR_EFFECTS.items():
        print(f"    rung {rung:3d} h pair ({ex:>3s} exposure): {eff:4.1f} pp  "
              f"-> sigma/effect = {sigma/eff:.2f}")

    # Runs per condition for 80% power, two-sided alpha=0.05, two-group contrast.
    # n = 2(z_{a/2}+z_b)^2 sigma^2 / delta^2 ; (1.959964+0.841621)^2 * 2 = 15.698
    K = 15.698
    from math import ceil
    print("\n  RUNS PER CONDITION for 80% power at alpha=0.05 (two-sided, two groups):")
    for rung, (ex, eff) in PAIR_EFFECTS.items():
        req = K * sigma ** 2 / eff ** 2
        # The formula is continuous and can return less than 2. Two per group is
        # the floor of the design, so that is what gets reported; the raw value
        # is kept beside it so the arithmetic stays checkable.
        line = (f"    to detect {eff:4.1f} pp: {max(2, int(ceil(req))):3d} runs "
                f"(raw {req:.2f}")
        if s_lo:
            req_lo = K * s_lo ** 2 / eff ** 2
            req_hi = K * s_hi ** 2 / eff ** 2
            line += f", CI {req_lo:.1f}-{req_hi:.1f})"
        else:
            line += ")"
        print(line)
    mde2 = sigma * (K / 2) ** 0.5
    mde3 = sigma * (K / 3) ** 0.5
    print(f"\n  Conversely, the smallest effect detectable at 2 runs per condition is "
          f"{mde2:.1f} pp,")
    print(f"  and at 3 runs per condition {mde3:.1f} pp.")
    print("\n  The practical recommendation the paper makes follows from this: two runs per")
    print("  condition will not resolve an effect of this size, but they DO reveal whether")
    print("  between-run spread rivals it -- which is the thing a single run cannot show.")

    pd.DataFrame([dict(sigma_pp=round(sigma, 2), sigma_raw_pp=round(sigma_raw, 2),
                       c4=round(_c4, 5), df=df,
                       sigma_lo=round(s_lo, 2) if s_lo else None,
                       sigma_hi=round(s_hi, 2) if s_hi else None,
                       rungs_with_replicates=len(contrib),
                       mde_at_2_runs=round(mde2, 2), mde_at_3_runs=round(mde3, 2))]
                 ).to_csv(f"{WORK}/stage2_sigma.csv", index=False)
    pd.DataFrame([dict(rung=r, n_runs=n, sd_pp=round(s, 2),
                       asr_min=round(lo, 1), asr_max=round(hi, 1))
                  for r, n, s, lo, hi in contrib]
                 ).to_csv(f"{WORK}/stage2_sigma_by_rung.csv", index=False)

# ---- verdict ---------------------------------------------------------------
verdict = None
if sigma is not None:
    worst = max(sigma / eff for _, (_, eff) in PAIR_EFFECTS.items() if eff)
    if worst < 0.33:
        verdict = "CLEAN"
        print("\n  VERDICT: between-run variance is small relative to every pair effect.")
        print("  The pairs stand. Report sigma as a measured bound and keep the")
        print("  data-volume claim as the headline.")
    elif worst < 0.75:
        verdict = "MATERIAL"
        print("\n  VERDICT: between-run variance is material for at least one pair but does")
        print("  not account for the largest. Report sigma beside every pair effect and")
        print("  state which pairs survive it.")
    else:
        verdict = "DOMINANT"
        print("\n  VERDICT: between-run variance is comparable to or larger than the effects")
        print("  attributed to data volume. This is the paper's current framing, now")
        print("  measured across rungs rather than inferred from one pair.")
    print(f"\n  verdict = {verdict}")

In [ ]:
# ---- refresh the ladder and the three pairs, if Task B is attached ---------
print("\n" + "=" * 74)
print("STAGE 2c  --  ladder and iso-WER pairs, refreshed from whatever is attached")
print("=" * 74)
TASKB = {"1h final":"eps50_rung1_final_T5_n100.csv",
         "5h final":"eps50_rung5_final_T5_n100.csv",
         "25h final":"eps50_rung25_final_T5_n100.csv",
         "100h ms-1000":"eps50_rung100_milestone-1000_T5_n100.csv",
         "100h ms-2000":"eps50_rung100_milestone-2000_T5_n100.csv",
         "100h ms-3500":"eps50_rung100_milestone-3500_T5_n100.csv"}
have = {k: FOUND[v] for k, v in TASKB.items() if v in FOUND}
if f"eps50_rung100-final_{EDIT_TYPE}_n{N_EVAL}.csv" in FOUND:
    have["100h final"] = FOUND[f"eps50_rung100-final_{EDIT_TYPE}_n{N_EVAL}.csv"]
_TRAJ = f"eps50_rung1-trajfinal_{EDIT_TYPE}_n{N_EVAL}.csv"
if _TRAJ in FOUND:
    have["1h s1337 run2"] = FOUND[_TRAJ]
# Replicates join the ladder view too, so the table shows every run of every rung
# rather than one arbitrarily chosen run per rung.
for _r in (1, 5, 25):
    for _s in (1338, 1339, 1340, 1341):
        _f = f"eps50_rung{_r}-s{_s}_{EDIT_TYPE}_n{N_EVAL}.csv"
        if _f in FOUND:
            have[f"{_r}h s{_s}"] = FOUND[_f]
print(f"  {len(have)} conditions available: {sorted(have)}")

lad = []
S = {}
for k, p in sorted(have.items()):
    s, e = load_eps(p); S[k] = (s, e)
    n = len(s); kk = sum(s.values())
    lad.append(dict(condition=k, n=n, asr=100.0*kk/n, censored=n-kk,
                    eps50_ok=med([e[u] for u in s if s[u]]),
                    eps50_imp=med(list(e.values()))))
if lad:
    L = pd.DataFrame(lad).sort_values("condition")
    print()
    print(L.to_string(index=False))
    L.to_csv(f"{WORK}/stage2_ladder.csv", index=False)

PAIRS = [(1, "1h final", "100h ms-1000", 40), (2, "5h final", "100h ms-2000", 16),
         (3, "25h final", "100h ms-3500", 4)]
rows = []
for k, lo, hi, ex in PAIRS:
    if lo not in S or hi not in S:
        print(f"\n  pair {k}: needs {lo} and {hi} -- not both attached, skipped")
        continue
    a, b = S[lo][0], S[hi][0]
    p, n10, n01, nc = mcnemar(a, b)
    da = 100.0*sum(b.values())/len(b) - 100.0*sum(a.values())/len(a)
    rows.append(dict(pair=k, exposure=f"{ex}x", low=lo, high=hi,
                     delta_pp=round(da, 1), mcnemar_p=p, n10=n10, n01=n01, n=nc))
if rows:
    P = pd.DataFrame(rows)
    print("\n  iso-WER pairs, exact McNemar on the shared utterances:")
    print(P.to_string(index=False))
    P.to_csv(f"{WORK}/stage2_pairs.csv", index=False)
    print("\n  These reproduce Table III. If they differ from -38 / -33 / -13 pp,")
    print("  something about the inputs changed -- find out what before editing the paper.")

In [ ]:
# ---- paste-ready block ----------------------------------------------------
print("\n" + "=" * 74)
print("STAGE 2d  --  paste this into RUNLOG.md")
print("=" * 74)
lines = [f"# {time.strftime('%Y-%m-%d')} — NB-16 replicate session",
         f"Wall clock {(time.time()-SESSION_START)/3600:.2f} h. "
         f"n={N_EVAL}, {EDIT_TYPE}, attack seed {SEED}, bisection {BISECT_ROUNDS} rounds, "
         f"probe {PROBE_STEPS} / confirm {CONFIRM_STEPS}, AMP {USE_AMP}, batch {BATCH_SIZE}.",
         f"Targets sha256 {PROV.get('targets_sha256','?')}",
         f"transformers {PROV.get('transformers','?')}", ""]
for r in (new_rows or []):
    lines.append(f"* **{r['tag']}** n={r['n']} ASR@2e-3 **{r['asr']:.1f}%** "
                 f"censored {r['censored']} imputed median {r['eps50_imp']:.4e}")

if 'RUNS' in dir() and RUNS:
    lines += ["", "**Runs visible per rung (ASR@2e-3):**"]
    for rung in sorted(RUNS):
        vals = ", ".join(f"{r['label']} {r['asr']:.1f}%" for r in RUNS[rung])
        lines.append(f"* rung {rung} h — {vals}")

if pair_rows:
    lines += ["", "**Within-rung pairwise McNemar (same condition, different run):**", "",
              "| rung | run A | run B | ASR A | ASR B | delta pp | p |",
              "|---|---|---|---|---|---|---|"]
    for r in pair_rows:
        lines.append(f"| {r['rung']} h | {r['run_a']} | {r['run_b']} | {r['asr_a']}% | "
                     f"{r['asr_b']}% | {r['delta_pp']:+} | {r['mcnemar_p']:.3e} |")

if sigma is not None:
    lines += ["", f"**Between-run sigma = {sigma:.2f} pp** (bias-corrected; raw "
                  f"pooled sd {sigma_raw:.2f}, df {int(den)}, {len(contrib)} rung(s) "
                  f"with >=2 runs)."]
    if req is not None:
        from math import ceil as _ceil
        _n = max(2, int(_ceil(15.698 * sigma ** 2 / 38.0 ** 2)))
        lines.append(f"Runs per condition for 80% power at the 38 pp pair-1 effect: "
                     f"{_n} (raw {15.698*sigma**2/38.0**2:.2f}).")
    lines.append(f"**Verdict: {verdict}.**")
else:
    lines += ["", "**No rung has two runs in this session's view.** Sigma not estimable; "
                  "attach the published eps50-*-n100 datasets and re-run Stage 2."]

for s in summary:
    if s["status"] not in ("complete", "skipped"):
        lines.append(f"* {s['tag']}: **{s['status']}** — publish and re-commit to continue.")
block = "\n".join(lines)
print(block)
open(f"{WORK}/RUNLOG_BLOCK.md", "w").write(block + "\n")

PROV["stage2_verdict"] = verdict
PROV["stage2_new"] = new_rows
PROV["stage2_sigma_pp"] = sigma
PROV["stage2_within_rung_pairs"] = pair_rows
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False, indent=2,
          default=str)
print("\n" + "=" * 74)
print("FILES WRITTEN")
print("=" * 74)
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nPublish /kaggle/working as 'eps50-replicates-n100'. CHECK THIS LIST FIRST:")
print("publishing replaces the dataset and deletes whatever a partial run did not")
print("regenerate. Then run NB-17 for the pooled refit and the power table.")

---

## After this notebook

1. If any condition reads `partial` or `not-started`: publish `/kaggle/working`, attach it back
   as an input, re-commit. Completed utterances are skipped, not redone.
2. When all four replicates are measured, run **NB-17**. It takes the CSVs from here plus
   `clean_wer_replicates.csv` from NB-15 and produces the run-clustered ANCOVA, the variance
   decomposition and the power table that go into the paper.

## Reading Stage 2b honestly

The `DOMINANT` verdict is the paper's current position, and the replicates are not there to
confirm it — they are there to put an interval on it. Three outcomes are all publishable:

- **σ stays near 30 pp.** The null stands, now with a measured spread at three rungs instead of
  an extrapolation from one pair. This is the strongest version of the current paper.
- **σ comes in much smaller**, and the 34 pp pair was an outlier. Then the pooled ANCOVA is
  worth refitting with run as a random effect, and the data-volume effect may well become
  identifiable again. The paper's framing would change, and that is a better paper.
- **σ comes in larger.** The null is firmer and the required-runs number rises.

What would be wrong is to run the replicates, dislike the number, and drop one. The pairwise
McNemar table prints every pair precisely so that selective reporting is visible.
